# 07 · Apps and real-time inference

Tasks run to completion; **apps** stay up and serve traffic. Same environment model
(image, resources, secrets), plus a URL, autoscaling, and rollout management. This is
Union's answer to "where does my FastAPI service / model server / dashboard live?"

**Learning goals**

1. Deploy a FastAPI app and call it; understand the deploy → activate lifecycle
2. Configure autoscaling (including scale-to-zero) and measure cold vs warm latency
3. Serve an LLM with the vLLM app environment (OpenAI-compatible endpoint)
4. Know the latency/throughput levers for production inference

> **Why the app code is in `scripts/`:** app deployment needs a deployable file — it isn't
> supported from interactive notebook sessions. The notebook *drives* the deploys and then
> exercises the endpoints. Open the scripts side-by-side as you go:
> [`fastapi_app.py`](./scripts/apps/fastapi_app.py) ·
> [`streamlit_app.py`](./scripts/apps/streamlit_app.py) ·
> [`vllm_app.py`](./scripts/apps/vllm_app.py)

In [ ]:
import flyte

flyte.init_from_config()

## 1. Deploy a FastAPI app

`scripts/apps/fastapi_app.py` defines a `FastAPIAppEnvironment`: the FastAPI object, an
image, resources, and a `Scaling` policy (0-2 replicas, scale-down after 5 idle minutes).
Deploying it returns a stable URL.

In [ ]:
!python scripts/apps/fastapi_app.py

In [ ]:
# Paste the URL printed above:
APP_URL = "https://REPLACE-ME"

import httpx

r = httpx.post(f"{APP_URL}/score", json={"text": "hello self-managed union"}, timeout=120)
r.json()

### Measure cold vs warm latency

With `replicas=(0, 2)` the first request after idle pays the **cold start** (pod schedule +
image pull + process boot). Subsequent requests hit a warm replica:

In [ ]:
import time

def probe(n: int = 10) -> list:
    times = []
    for i in range(n):
        t0 = time.perf_counter()
        httpx.post(f"{APP_URL}/score", json={"text": f"probe-{i}"}, timeout=120)
        times.append((time.perf_counter() - t0) * 1000)
    return [round(t, 1) for t in times]

latencies = probe()
print(f"first (cold-ish): {latencies[0]} ms")
print(f"warm p50: {sorted(latencies[1:])[len(latencies)//2]} ms")
latencies

**The production trade-off in one line:** `replicas=(0, N)` minimizes cost, `replicas=(1, N)`
(or more) buys away the cold start. For latency-sensitive inference keep min ≥ 1 and use
`scaledown_after` to control how aggressively extra replicas retire.

Other levers on `AppEnvironment` worth knowing:

| Lever | Effect |
|---|---|
| `scaling=flyte.app.Scaling(replicas=(min, max), scaledown_after=...)` | Replica autoscaling |
| `requires_auth=True` (default) | Endpoint requires Union auth; `False` makes it public inside your network |
| `parameters=[Parameter(...)]` | Feed model artifacts from task outputs (`RunOutput`) into the app at deploy time |
| `env_vars`, `secrets` | Same as tasks |
| `domain`/custom domains | Serve under your own hostname (platform config) |

Manage lifecycle from the CLI:

```bash
flyte get app                       # list apps + status
flyte deploy ...                    # ship a new version
```
Apps roll out per-version; deactivating stops traffic without deleting history.

## 2. Model-artifact wiring (train → serve)

The pattern that connects the batch world to serving: a training task returns a model
`File`; the app declares a `Parameter` with `RunOutput(task_name=...)` so each deploy
resolves the **latest** (or pinned) trained model, downloads it, and exposes it to the app
via an env var — no manual copying. See the *local-to-remote model serving* section of the
FastAPI app docs; the sketch:

```python
serving_env = FastAPIAppEnvironment(
    name="predictor",
    app=app,
    parameters=[
        Parameter(
            name="model",
            value=RunOutput(task_name="pipeline.train_model", type="file"),
            download=True,
            env_var="MODEL_PATH",       # app reads this at startup
        ),
    ],
    ...
)
```

## 3. LLM serving with vLLM

`VLLMAppEnvironment` wraps the vLLM server: point it at a HuggingFace model, give it a GPU,
and you get an **OpenAI-compatible** endpoint. The workshop default is a 0.5B model that
fits any L4/T4-class GPU.

> 💬 **Discuss:** for each service you plan to run, what does a cold start cost you (user-facing latency) vs an always-warm minimum replica (steady spend)? Where does each land on `replicas=(0, N)` vs `(1, N)`?

In [ ]:
# Requires a free GPU in your node pools. HF_MODEL_ID comes from .env (default Qwen2.5-0.5B).
!python scripts/apps/vllm_app.py

In [ ]:
LLM_URL = "https://REPLACE-ME"      # from the cell above

r = httpx.post(
    f"{LLM_URL}/v1/chat/completions",
    json={
        "model": "workshop-llm",
        "messages": [{"role": "user", "content": "One sentence: what is a dataplane?"}],
        "max_tokens": 60,
    },
    timeout=300,
)
r.json()["choices"][0]["message"]["content"]

Inference tuning levers (all on the app environment / vLLM args):

- **Throughput:** vLLM batches continuously — raise `--max-num-seqs`, use
  `--tensor-parallel-size N` with `gpu="L40s:N"` for models that need multiple GPUs
- **Cold start:** model download dominates — `disk` sizing + **model prefetching**
  (see `serve-and-deploy-apps/prefetching-models` in the docs) pre-stage weights
- **Cost:** `replicas=(0, 1)` parks the GPU when idle; the first request pays ~minutes
  of model load, so keep min=1 for interactive use

## 4. Dashboards: Streamlit

Any process that binds a port can be an app (`args=[...]` + `port=...`). The Streamlit
example ships a small dashboard the same way:

> 💬 **Discuss:** which GPU types do your node pools actually offer today, and which models on your roadmap fit them? (This decides `gpu=` for every serving app.)

In [ ]:
!python scripts/apps/streamlit_app.py

## 5. Apps calling tasks (and vice versa)

- **Task → app:** just HTTP (`httpx` from any task).
- **App → task:** an app can launch runs — e.g. a "retrain now" button:

```python
@app.post("/retrain")
async def retrain():
    run = flyte.with_servecontext().run(train_model, examples=1000, epochs=5)
    return {"run_url": run.url}
```

That closes the loop: batch pipelines (01-06) produce models; apps serve them; apps can
kick off new pipeline runs.

## Further reading

- Next: [08-v1-to-v2-migration](./08-v1-to-v2-migration.ipynb)
- Union docs → User guide → *Build apps*, *Configure apps*, *Serve and deploy apps*